# CODI global-head ablations
Runs on the frozen official GPT-2 CODI checkpoint. Only the output head is trained.

**Core:** unconstrained / fixed U28 / fixed random-28 at ranks 32, 64, 96;
first-token-only / full-trajectory fitting with matched state counts.
**Optional:** weight-only SVD, loss terms, matched-budget recovery, data efficiency,
and frozen-head SVAMP evaluation. Model architecture transfer is a separate experiment.

U28 is refitted from training first-token states (zero-based PCs 4:32). This tests the
same construction with clean splits; it is not the exact historical U28 artifact.
All choices are locked before test decoding. GSM8K has informed prior hypotheses, so
these are follow-up ablations, not a new untouched confirmatory benchmark.

Upload this notebook directly: it embeds its helper implementation and clones immutable
base commit `6a8d2e61950c67f012d0a9ba13ec8a70f3a25019`. No branch push is required.
No dataset attachments or historical result files are required. Model weights and
training/evaluation data download automatically. Checkpoint hashes and decoder parity
are checked in this run, and the dense baseline is evaluated alongside the ablations.
Enable Internet and a GPU, then run all cells. `SMOKE=True` is an optional quick check;
the default `SMOKE=False` runs the real experiment.

In [ ]:
SUITE = "core"  # core, initialization, losses, recovery, data, or all
SMOKE = False  # smoke output is explicitly non-scientific; never compare with full runs
SEEDS = [89]   # use [89, 90, 91] for final fitting-seed uncertainty
EVAL_DATASETS = ["gsm8k"]  # optionally add "svamp"; no refitting on evaluation data
RESUME_ROOT = ""  # optional prior output run directory containing manifest.json
OUTPUT_ROOT = "/kaggle/working/codi_global_head_ablations"
GENERATION_BATCH_SIZE = 8  # lower to 4 if GPU memory is tight
DISTILL_BATCH_SIZE = 8
MAX_NEW_TOKENS = 64
RANKS = (32, 64, 96)
FIT_QUESTIONS = 1024
SELECT_QUESTIONS = 256
RECOVERY_QUESTIONS = 256
CLEAN_EPOCHS = 4
RECOVERY_EPOCHS = 2
LEARNING_RATE = 2e-4
DATA_SIZES = [128, 256, 512, 1024]
BOOTSTRAP_SAMPLES = 5000
SPLIT_SEED = 20260909  # fixed across fitting seeds
SEED = SPLIT_SEED
if SMOKE:
    FIT_QUESTIONS, SELECT_QUESTIONS, RECOVERY_QUESTIONS = 48, 8, 8
    CLEAN_EPOCHS = RECOVERY_EPOCHS = 1
    DATA_SIZES = [36, 48]
    BOOTSTRAP_SAMPLES = 100
assert SUITE in {"core", "initialization", "losses", "recovery", "data", "all"}
assert SEEDS and len(SEEDS) == len(set(SEEDS))
assert set(EVAL_DATASETS) <= {"gsm8k", "svamp"} and "gsm8k" in EVAL_DATASETS

import copy, glob, hashlib, json, os, pathlib, random, shutil, subprocess, sys, time
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "6a8d2e61950c67f012d0a9ba13ec8a70f3a25019"
REPO_DIR = "/kaggle/working/latent-reasoning"
if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", RUN_COMMIT], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
CODE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert CODE_COMMIT == RUN_COMMIT
print("Base implementation:", CODE_COMMIT)

## Environment and frozen model

In [ ]:
PINNED_PACKAGES = {
    "transformers": "4.52.4",
    "peft": "0.15.2",
    "datasets": "3.6.0",
    "huggingface_hub": "0.32.4",
}
from importlib.metadata import PackageNotFoundError, version as package_version

def installed_package_version(name):
    try:
        return package_version(name)
    except PackageNotFoundError:
        return None

missing = [f"{name}=={wanted}" for name, wanted in PINNED_PACKAGES.items()
           if installed_package_version(name) != wanted]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# PEFT 0.15 probes torchao when it is installed. Kaggle images sometimes contain an
# old incompatible torchao; uninstalling that optional package is safer than allowing
# adapter construction to fail.
probe = (
    "from peft.import_utils import is_torchao_available\n"
    "try:\n    print('ok' if is_torchao_available() else 'absent')\n"
    "except ImportError as error:\n    print('incompatible:' + str(error))\n"
)
state = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
state = (state.stdout + state.stderr).strip()
print("torchao:", state)
if state.startswith("incompatible"):
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)

subprocess.run([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_global_low_rank_head.py",
    "tests/test_official_codi.py",
], check=True)

In [ ]:
import torch
import torch.nn as nn
from src.mech.global_low_rank_head import (
    NestedLowRankVocabularyHead,
    activation_whitened_factors,
    distil_nested_head,
    evaluate_nested_head,
)
from src.mech.eigenspace_readout import benchmark_vocabulary_head
from src.inference.official_codi_fast import (
    generate_official_codi_fast, prepare_official_codi_batches,
)
from src.models.official_codi import (
    build_official_codi_gpt2, download_official_checkpoint,
    generate_official_codi, load_official_checkpoint, resolve_torch_dtype,
)
from src.utils.config import load_config

torch.manual_seed(SEED)
random.seed(SEED)
cfg = load_config("configs/official_codi_gpt2.yaml")
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
device = torch.device("cuda")
dtype = resolve_torch_dtype("float32", device)
checkpoint = download_official_checkpoint(
    repo_id=str(cfg.checkpoint.repo_id), revision=str(cfg.checkpoint.revision),
    filename=str(cfg.checkpoint.filename), expected_sha256=str(cfg.checkpoint.sha256),
    token=os.environ.get("HF_TOKEN") or None,
)
model, tokenizer = build_official_codi_gpt2(
    base_model=str(cfg.model.base_model), base_revision=str(cfg.model.base_revision),
    dtype=dtype, settings=cfg.model, token=os.environ.get("HF_TOKEN") or None,
)
load_report = load_official_checkpoint(
    model, checkpoint, expected_sha256=str(cfg.checkpoint.sha256)
)
for parameter in model.parameters():
    parameter.requires_grad_(False)
model.to(device=device, dtype=dtype).eval()
base_model = model.codi.get_base_model()
full_head = base_model.get_output_embeddings()
hidden_size = int(model.config.hidden_size)
vocabulary_size = int(model.eot_id)
readout_weight = full_head.weight[:vocabulary_size].detach()
readout_bias = (
    None if getattr(full_head, "bias", None) is None
    else full_head.bias[:vocabulary_size].detach()
)
assert hidden_size == 768
print({"checkpoint": load_report.checkpoint_sha256,
       "hidden": hidden_size, "vocabulary": vocabulary_size})

## Embedded ablation helpers

In [ ]:
from __future__ import annotations
ABLATION_SOURCE_SHA256 = '6c0f3d8e2655d3b8fbae3b36ffeae76d0bd2092a547187992916f6a713e08e09'
"""Controlled initializers for the CODI global-head ablation notebook."""

import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils import parametrize

from src.mech.global_low_rank_head import (
    NestedLowRankVocabularyHead, activation_whitened_factors,
)


class FixedSubspaceRows(nn.Module):
    """Keep the first k rows fixed and every learned row orthogonal to them."""

    def __init__(self, basis):
        super().__init__()
        self.register_buffer("basis", basis.detach().clone())

    def forward(self, weight):
        residual = weight[self.basis.shape[1]:]
        residual = residual - (residual @ self.basis) @ self.basis.T
        return torch.cat((self.basis.T, residual), dim=0)


def colon_basis(states, start=4, stop=32):
    """PCA band using training first-token states only; indices are zero based."""
    if len(states) <= stop:
        raise ValueError("Need more first-token training states than the PCA stop index")
    centered = states.double() - states.double().mean(0)
    _, vectors = torch.linalg.eigh(centered.T @ centered / (len(states) - 1))
    return vectors.flip(1)[:, start:stop].to(states)


@torch.no_grad()
def initialize_head(states, weight, ranks=(32, 64, 96), *, bias=None,
                    seed=0, initialization="whitened", fixed_basis=None):
    """Initialize equal-total-rank heads; fixed coordinates replace learned slots.

    Constrained heads fit a whitened residual in the orthogonal complement and
    jointly regress vocabulary coefficients on the resulting coordinates. Their
    down-projection constraint remains exact during optimization.
    """
    rank = max(ranks)
    device = weight.device
    states = states.to(device=device, dtype=weight.dtype)
    centre = states.mean(0)
    output_bias = F.linear(centre, weight, bias)
    if fixed_basis is not None:
        basis = fixed_basis.to(weight)
        if basis.shape[1] >= min(ranks):
            raise ValueError("Every rank must leave room for learned residual directions")
        if not torch.allclose(basis.T @ basis, torch.eye(basis.shape[1], device=device),
                              atol=2e-5, rtol=2e-5):
            raise ValueError("Fixed basis must be orthonormal")
        k = basis.shape[1]
        residual_states = states - (states @ basis) @ basis.T
        residual_weight = weight - (weight @ basis) @ basis.T
        _, down, _, _, _ = activation_whitened_factors(
            residual_states, residual_weight, rank-k, seed=seed, compute_device=device)
        down = down - (down @ basis) @ basis.T
        # Normalize the residual row span for a well-conditioned coordinate fit.
        down = torch.linalg.qr(down.T, mode="reduced").Q.T
        down = torch.cat((basis.T, down), dim=0)
        centered = states - centre
        coordinates = centered @ down.T
        gram = coordinates.T @ coordinates
        ridge = max(float(gram.diagonal().mean()) * 1e-6, 1e-8)
        # Avoid allocating [number of states, vocabulary] teacher logits here.
        rhs = (coordinates.T @ centered) @ weight.T
        up = torch.linalg.solve(gram + ridge * torch.eye(rank, device=device), rhs).T
    elif initialization == "whitened":
        centre, down, up, output_bias, _ = activation_whitened_factors(
            states, weight, rank, readout_bias=bias, seed=seed, compute_device=device)
    elif initialization == "weight_svd":
        # Randomized weight-only SVD with the same oversampling/power count.
        with torch.random.fork_rng(devices=[device.index or 0] if device.type == "cuda" else []):
            torch.manual_seed(seed)
            left, singular, right = torch.svd_lowrank(
                weight, q=min(rank + 16, min(weight.shape)), niter=1)
        down = right[:, :rank].T
        up = left[:, :rank] * singular[:rank]
    else:
        raise ValueError(f"Unknown initialization: {initialization}")
    head = NestedLowRankVocabularyHead.from_whitened_factors(
        centre, down, up, output_bias, ranks)
    if fixed_basis is not None:
        parametrize.register_parametrization(head.down, "weight", FixedSubspaceRows(basis))
        # Bias shifts do not change the fixed row span. All heads train their biases.
    return head


def matched_state_sample(states, count, seed):
    if count <= 0 or count > len(states):
        raise ValueError("Invalid matched-state count")
    order = torch.randperm(len(states), generator=torch.Generator().manual_seed(seed))
    return states[order[:count]]


def paired_interval(reference, candidate, *, samples=5000, seed=0):
    reference = torch.as_tensor(reference, dtype=torch.bool)
    candidate = torch.as_tensor(candidate, dtype=torch.bool)
    if reference.shape != candidate.shape or reference.numel() == 0:
        raise ValueError("Paired flags must have matching nonempty shapes")
    differences = candidate.float() - reference.float()
    generator = torch.Generator().manual_seed(seed)
    estimates = []
    for start in range(0, samples, 250):
        indices = torch.randint(len(reference), (min(250, samples-start), len(reference)),
                                generator=generator)
        estimates.append(differences[indices].mean(1))
    values = torch.cat(estimates)
    return {"delta_pp": 100 * float(differences.mean()),
            "ci95_low_pp": 100 * float(values.quantile(.025)),
            "ci95_high_pp": 100 * float(values.quantile(.975)),
            "correct_to_wrong": int((reference & ~candidate).sum()),
            "wrong_to_correct": int((~reference & candidate).sum())}


## Lock the experiment grid and output identity
Each suite includes its own reference. Core heads receive clean distillation only;
recovery is isolated in its own suite. All ranks are nested prefixes trained together,
not independent rank-specific fits. Fixed-28 heads have 4/36/68 learned residual
coordinates at ranks 32/64/96. Random-28 uses the same initializer and constraint.
The random basis changes with the fitting seed.

Coverage compares the same training questions and the same number of states; the
all-trajectory arm samples one observed position per question. Data-size
arms use nested question subsets and matched state presentations/optimizer updates;
smaller pools replay states so every available question is still represented.
Recovery continuation arms start from identical clean weights, use identical pool
sizes and epoch counts, and select checkpoints using validation only.

In [ ]:
def make_grid():
    arms = [{"name": "baseline"}]
    if SUITE in {"core", "all"}:
        arms += [{"name": "fixed_u28", "fixed": "u28"},
                 {"name": "fixed_random28", "fixed": "random"},
                 {"name": "coverage_first", "coverage": "first"},
                 {"name": "coverage_all_matched", "coverage": "all_matched"}]
    if SUITE in {"initialization", "all"}:
        arms += [{"name": "weight_svd", "initialization": "weight_svd"}]
    if SUITE in {"losses", "all"}:
        arms += [{"name": "no_margin", "margin_weight": 0.0},
                 {"name": "no_top_token", "token_weight": 0.0}]
    if SUITE in {"recovery", "all"}:
        arms += [{"name": "recovery_" + kind, "recovery": kind}
                 for kind in ("repeat_clean", "teacher", "onpolicy")]
    if SUITE in {"data", "all"}:
        arms += [{"name": f"data_{n}", "questions": n} for n in DATA_SIZES]
    return arms
ARMS = make_grid()
LOCKED_CONFIG = dict(suite=SUITE, smoke=SMOKE, seeds=SEEDS, ranks=RANKS,
    fit=FIT_QUESTIONS, selection=SELECT_QUESTIONS, recovery=RECOVERY_QUESTIONS,
    clean_epochs=CLEAN_EPOCHS, recovery_epochs=RECOVERY_EPOCHS,
    lr=LEARNING_RATE, generation_batch=GENERATION_BATCH_SIZE,
    distill_batch=DISTILL_BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
    split_seed=SPLIT_SEED, data_sizes=DATA_SIZES, datasets=EVAL_DATASETS,
    bootstrap_samples=BOOTSTRAP_SAMPLES, arms=ARMS, base_commit=CODE_COMMIT,
    ablation_source=ABLATION_SOURCE_SHA256, checkpoint=load_report.checkpoint_sha256,
    torch_version=torch.__version__, cuda_version=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0))
manifest_text = json.dumps(LOCKED_CONFIG, sort_keys=True, indent=2)
RUN_ID = hashlib.sha256(manifest_text.encode()).hexdigest()[:16]
RUN_DIR = pathlib.Path(OUTPUT_ROOT) / (('smoke_' if SMOKE else 'full_') + RUN_ID)
RUN_DIR.mkdir(parents=True, exist_ok=True)
if RESUME_ROOT:
    prior = pathlib.Path(RESUME_ROOT)
    assert (prior / 'manifest.json').read_text() == manifest_text, 'Resume configuration mismatch'
    shutil.copytree(prior, RUN_DIR, dirs_exist_ok=True)
manifest_path = RUN_DIR / 'manifest.json'
if manifest_path.exists():
    assert manifest_path.read_text() == manifest_text
manifest_path.write_text(manifest_text)

def save_json(path, value):
    tmp = pathlib.Path(str(path) + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, default=str))
    tmp.replace(path)

def save_pt(path, value):
    tmp = pathlib.Path(str(path) + '.tmp')
    torch.save(value, tmp)
    tmp.replace(path)

print('LOCKED:', RUN_DIR)
print('Fits per seed:', len(ARMS), '| head evaluations per dataset:', len(ARMS)*len(RANKS)*len(SEEDS))
print(json.dumps(ARMS, indent=2))

## Unique-question partitions (training data only)

In [ ]:
from src.data.datasets import load_train_set, load_eval_set
training_rows = load_train_set(load_config('configs/data.yaml'), trace_style='eq_only')
def normalize_question(text):
    return ' '.join(str(text).casefold().split())
unique = {}
for row in training_rows:
    unique.setdefault(normalize_question(row['question']), str(row['question']))
unique_questions = list(unique.values())
order = torch.randperm(len(unique_questions), generator=torch.Generator().manual_seed(SPLIT_SEED)).tolist()
required = FIT_QUESTIONS + SELECT_QUESTIONS + RECOVERY_QUESTIONS
assert required <= len(order)
chosen = [unique_questions[i] for i in order[:required]]
fit_questions = chosen[:FIT_QUESTIONS]
select_questions = chosen[FIT_QUESTIONS:FIT_QUESTIONS + SELECT_QUESTIONS]
recovery_questions = chosen[FIT_QUESTIONS + SELECT_QUESTIONS:]
assert len({normalize_question(q) for q in chosen}) == required
save_json(RUN_DIR / 'partitions.json', dict(fit=fit_questions, selection=select_questions,
                                         recovery=recovery_questions))
print(dict(fit=len(fit_questions), selection=len(select_questions), recovery=len(recovery_questions)))

In [ ]:
parity_questions = select_questions[:16]
reference_parity = generate_official_codi(
    model, tokenizer, parity_questions,
    latent_iterations=int(cfg.eval.latent_iterations),
    max_new_tokens=MAX_NEW_TOKENS, batch_size=GENERATION_BATCH_SIZE,
    device=device, answer_cue="The answer is:", force_answer_cue=True,
)
parity_prepared = prepare_official_codi_batches(
    tokenizer, parity_questions, batch_size=GENERATION_BATCH_SIZE,
    length_bucketed=False,
)
fast_parity = generate_official_codi_fast(
    model, tokenizer, parity_prepared,
    latent_iterations=int(cfg.eval.latent_iterations),
    max_new_tokens=MAX_NEW_TOKENS, device=device, answer_cue="The answer is:",
)
assert tuple(reference_parity) == fast_parity.texts, (
    "The transformer-body fast path changed decoded outputs; stop before fitting."
)
print({"fastpath_parity_examples": len(reference_parity), "exact": True})

## Collect and cache teacher trajectories
Capture question IDs and token positions, including the termination decision.
Cached states are on CPU. The original head is restored even if collection fails.

In [ ]:
@torch.no_grad()
def collect_bundle(questions, head, tag):
    path = RUN_DIR / (tag + '.pt')
    if path.exists():
        return torch.load(path, map_location='cpu', weights_only=False)
    states, positions, question_ids = [], [], []
    base_model.set_output_embeddings(head)
    try:
        # Explicit chunks preserve the mapping from active rows to question IDs.
        for start in range(0, len(questions), GENERATION_BATCH_SIZE):
            chunk = questions[start:start + GENERATION_BATCH_SIZE]
            def observer(hidden, active_mask, answer_position):
                active = active_mask.detach().cpu().bool()
                states.append(hidden[active_mask].detach().cpu().float())
                positions.extend([int(answer_position)] * int(active.sum()))
                question_ids.extend((torch.arange(len(chunk))[active] + start).tolist())
            prepared = prepare_official_codi_batches(tokenizer, chunk,
                batch_size=GENERATION_BATCH_SIZE, length_bucketed=False)
            generate_official_codi_fast(model, tokenizer, prepared,
                latent_iterations=int(cfg.eval.latent_iterations), max_new_tokens=MAX_NEW_TOKENS,
                device=device, answer_cue='The answer is:', answer_state_observer=observer)
            if start % (GENERATION_BATCH_SIZE * 16) == 0:
                print(tag, start, '/', len(questions), flush=True)
    finally:
        base_model.set_output_embeddings(full_head)
    bundle = dict(states=torch.cat(states), positions=torch.tensor(positions),
                  question_ids=torch.tensor(question_ids), questions=list(questions))
    assert len(bundle['states']) == len(bundle['positions']) == len(bundle['question_ids'])
    save_pt(path, bundle)
    return bundle

fit_bundle = collect_bundle(fit_questions, full_head, 'teacher_fit')
select_bundle = collect_bundle(select_questions, full_head, 'teacher_selection')
fit_states, select_states = fit_bundle['states'], select_bundle['states']
first_states = fit_states[fit_bundle['positions'] == 0]
assert len(first_states) == FIT_QUESTIONS
U28 = colon_basis(first_states)
save_pt(RUN_DIR / 'u28_train_only.pt', dict(basis=U28, questions=fit_questions, band=[4, 32]))
# Match optimizer updates while retaining every question in each data subset.
# Smaller subsets replay their states; larger subsets introduce new states.
DATA_STATE_BUDGET = len(fit_states)
print('States:', len(fit_states), len(select_states), '| data budget:', DATA_STATE_BUDGET)

## Fit all locked arms before loading test labels

In [ ]:
from dataclasses import asdict

def arm_states(arm, seed):
    if arm.get('coverage') == 'first':
        return first_states
    if arm.get('coverage') == 'all_matched':
        generator = torch.Generator().manual_seed(seed)
        indices = []
        for question in range(FIT_QUESTIONS):
            eligible = torch.where(fit_bundle['question_ids'] == question)[0]
            indices.append(eligible[torch.randint(len(eligible), (1,), generator=generator)].item())
        return fit_states[indices]
    if 'questions' in arm:
        available = fit_states[fit_bundle['question_ids'] < arm['questions']]
        order = torch.randperm(len(available), generator=torch.Generator().manual_seed(seed))
        order = order.repeat((DATA_STATE_BUDGET + len(available) - 1) // len(available))[:DATA_STATE_BUDGET]
        return available[order]
    return fit_states

def new_head(arm, states, seed):
    torch.manual_seed(seed)
    basis = None
    if arm.get('fixed') == 'u28':
        basis = U28
    elif arm.get('fixed') == 'random':
        basis = torch.linalg.qr(torch.randn(hidden_size, 28,
            generator=torch.Generator().manual_seed(seed)), mode='reduced').Q
    return initialize_head(states, readout_weight, RANKS, bias=readout_bias, seed=seed,
        initialization=arm.get('initialization', 'whitened'), fixed_basis=basis).to(device)

def fit_head(head, states, arm, seed, epochs):
    return asdict(distil_nested_head(head, states, select_states, readout_weight,
        readout_bias=readout_bias, epochs=epochs, batch_size=DISTILL_BATCH_SIZE,
        learning_rate=LEARNING_RATE, temperature=2.0, kl_weight=1.0,
        token_weight=arm.get('token_weight', .25), margin_weight=arm.get('margin_weight', .25),
        nested_weight=.5, minimum_margin=.25, anchor_strength=1e-5, seed=seed))

def position_metrics(head):
    result = {}
    for rank in RANKS:
        result[str(rank)] = {}
        for label, mask in [('all', torch.ones(len(select_states), dtype=torch.bool)),
                            ('p0', select_bundle['positions'] == 0),
                            ('p1', select_bundle['positions'] == 1),
                            ('p2plus', select_bundle['positions'] >= 2)]:
            count = int(mask.sum())
            result[str(rank)][label] = dict(states=count, **(evaluate_nested_head(
                head, select_states[mask], readout_weight, readout_bias=readout_bias,
                rank=rank, batch_size=DISTILL_BATCH_SIZE) if count else {}))
    return result

for seed in SEEDS:
    for arm in ARMS:
        name = arm['name']
        path = RUN_DIR / f'head_{seed}_{name}.pt'
        if path.exists():
            print('Resume completed fit:', seed, name, flush=True)
            continue
        print('Fitting:', seed, name, flush=True)
        states = arm_states(arm, seed)
        head = new_head(arm, states, seed)
        initial = position_metrics(head)
        if 'recovery' in arm:
            baseline = torch.load(RUN_DIR / f'head_{seed}_baseline.pt', map_location='cpu', weights_only=False)
            head.load_state_dict(baseline['state_dict'])
            teacher_extra = collect_bundle(recovery_questions, full_head, 'teacher_recovery')['states']
            kind = arm['recovery']
            head.set_rank(64)
            if kind == 'onpolicy':
                extra = collect_bundle(recovery_questions, head, f'onpolicy_{seed}')['states']
            elif kind == 'teacher':
                extra = teacher_extra
            else:
                extra = fit_states
            # Fixed additional-state count for every continuation, independent of arm.
            count = RECOVERY_QUESTIONS
            extra = matched_state_sample(extra, count, seed)
            states = torch.cat((fit_states, extra))
            initial = position_metrics(head)
            history = fit_head(head, states, arm, seed + 1, RECOVERY_EPOCHS)
        else:
            history = fit_head(head, states, arm, seed, CLEAN_EPOCHS)
        payload = dict(arm=arm, seed=seed, train_states=len(states),
            initial=initial, final=position_metrics(head), history=history,
            state_dict={k:v.detach().cpu().clone() for k,v in head.state_dict().items()})
        if arm.get('fixed'):
            rows = head.down.weight.detach()
            basis = rows[:28].T
            assert torch.max(torch.abs(rows[28:] @ basis)) < 2e-4
        save_pt(path, payload)
        save_json(path.with_suffix('.json'), {k:v for k,v in payload.items() if k != 'state_dict'})
        del head, payload
        torch.cuda.empty_cache()
        print('Saved:', path, flush=True)
print('All fits frozen. Test evaluation follows.')

## Locked full-answer evaluation and paired comparisons
Every head generates its own complete answer. Each completed arm is saved immediately;
an interrupted arm restarts, while completed arms are skipped on resume. Results include
per-question outputs, accuracy, baseline-correct to wrong flips, and paired bootstrap
intervals. Intervals are descriptive, unadjusted for multiple comparisons. No speed
claim is made from these single-run timings.

In [ ]:
from src.data.answer_extract import answers_match
import pandas as pd

@torch.no_grad()
def evaluate_generation(head, examples, path):
    if path.exists():
        return json.loads(path.read_text())
    base_model.set_output_embeddings(head)
    records = []
    started = time.perf_counter()
    try:
        for start in range(0, len(examples), GENERATION_BATCH_SIZE):
            chunk = examples[start:start + GENERATION_BATCH_SIZE]
            prepared = prepare_official_codi_batches(tokenizer, [str(x['question']) for x in chunk],
                batch_size=GENERATION_BATCH_SIZE, length_bucketed=False)
            generated = generate_official_codi_fast(model, tokenizer, prepared,
                latent_iterations=int(cfg.eval.latent_iterations), max_new_tokens=MAX_NEW_TOKENS,
                device=device, answer_cue='The answer is:')
            for offset, (example, text, tokens) in enumerate(zip(chunk, generated.texts, generated.token_ids)):
                records.append(dict(index=start+offset, question=str(example['question']),
                    gold=str(example['gold']), text=text, token_ids=list(tokens),
                    correct=bool(answers_match(text, example['gold']))))
    finally:
        base_model.set_output_embeddings(full_head)
    assert len(records) == len(examples)
    flags = [r['correct'] for r in records]
    result = dict(examples=len(flags), correct=sum(flags), accuracy=sum(flags)/len(flags),
                  flags=flags, seconds_single_run=time.perf_counter()-started, records=records)
    save_json(path, result)
    return result

rows, comparisons = [], []
for dataset in EVAL_DATASETS:
    examples = load_eval_set(dataset, load_config(cfg.data_config).eval[dataset])
    if dataset == 'gsm8k':
        assert len(examples) == 1319
    selected_overlap = {normalize_question(x['question']) for x in examples} & {normalize_question(q) for q in chosen}
    assert not selected_overlap, 'Evaluation question overlaps a fitting/selection/recovery question'
    if SMOKE:
        examples = examples[:8]
    dense = evaluate_generation(full_head, examples, RUN_DIR / f'eval_{dataset}_dense.json')
    rows.append(dict(dataset=dataset, seed=-1, arm='dense', rank=768,
                     accuracy=dense['accuracy'], correct=dense['correct'], retention=1.0))
    results = {}
    for seed in SEEDS:
        for arm in ARMS:
            payload = torch.load(RUN_DIR / f'head_{seed}_{arm["name"]}.pt', map_location='cpu', weights_only=False)
            head = new_head(arm, arm_states(arm, seed), seed)
            head.load_state_dict(payload['state_dict'])
            head.eval()
            for rank in RANKS:
                head.disable_adaptive()
                head.set_rank(rank)
                result = evaluate_generation(head, examples,
                    RUN_DIR / f'eval_{dataset}_{seed}_{arm["name"]}_r{rank}.json')
                results[(seed, arm['name'], rank)] = result
                rows.append(dict(dataset=dataset, seed=seed, arm=arm['name'], rank=rank,
                    accuracy=result['accuracy'], correct=result['correct'],
                    retention=result['accuracy']/dense['accuracy'] if dense['accuracy'] else None,
                    **paired_interval(dense['flags'], result['flags'], samples=BOOTSTRAP_SAMPLES, seed=seed)))
                pd.DataFrame(rows).to_csv(RUN_DIR / 'results.csv', index=False)
                print(dataset, seed, arm['name'], rank, result['correct'], '/', len(examples), flush=True)
            del head, payload
            torch.cuda.empty_cache()
    # Compare matched ablation arms directly, not only each arm against dense.
    pairs = [('baseline', 'fixed_u28'), ('fixed_random28', 'fixed_u28'),
             ('coverage_all_matched', 'coverage_first'), ('baseline', 'weight_svd'),
             ('baseline', 'no_margin'), ('baseline', 'no_top_token'),
             ('baseline', 'recovery_repeat_clean'), ('recovery_repeat_clean', 'recovery_teacher'),
             ('recovery_teacher', 'recovery_onpolicy')]
    pairs += [(f'data_{max(DATA_SIZES)}', f'data_{n}') for n in DATA_SIZES[:-1]]
    for seed in SEEDS:
        for reference, candidate in pairs:
            for rank in RANKS:
                a, b = (seed, reference, rank), (seed, candidate, rank)
                if a in results and b in results:
                    comparisons.append(dict(dataset=dataset, seed=seed, rank=rank,
                        reference=reference, candidate=candidate,
                        **paired_interval(results[a]['flags'], results[b]['flags'],
                                          samples=BOOTSTRAP_SAMPLES, seed=seed)))
        if (seed, 'fixed_u28', 64) in results:
            comparisons.append(dict(dataset=dataset, seed=seed, rank='96_vs_64',
                reference='baseline_r96', candidate='fixed_u28_r64',
                **paired_interval(results[(seed, 'baseline', 96)]['flags'],
                                  results[(seed, 'fixed_u28', 64)]['flags'],
                                  samples=BOOTSTRAP_SAMPLES, seed=seed)))
    pd.DataFrame(comparisons).to_csv(RUN_DIR / 'paired_comparisons.csv', index=False)
results_frame = pd.DataFrame(rows)
display(results_frame)
seed_summary = results_frame[results_frame.arm != 'dense'].groupby(['dataset', 'arm', 'rank']).accuracy.agg(['count', 'mean', 'std'])
seed_summary.to_csv(RUN_DIR / 'seed_summary.csv')
display(seed_summary)
save_json(RUN_DIR / 'completion.json', dict(complete=True, smoke=SMOKE, run_id=RUN_ID))
print('Download this output directory:', RUN_DIR)

## Reading and resuming results
- `results.csv`: per-seed, per-rank accuracy and paired differences versus dense.
- `paired_comparisons.csv`: direct controlled comparisons, including U28 rank 64 versus baseline rank 96.
- `seed_summary.csv`: mean and fitting-seed standard deviation (blank with one seed).
- `head_*.json`: train-state counts, epoch history, initial/final agreement and KL by position.
- `head_*.pt`: trained weights; `eval_*.json`: per-question predictions and paired flags.
- `manifest.json`, `partitions.json`, `u28_train_only.pt`: provenance and split/basis records.

For another session, attach the prior output as a Kaggle input and set `RESUME_ROOT` to
its exact run folder. Keep configuration, seeds, helper source and runtime identical.
Same-session reruns reuse the matching run folder automatically. Change `SUITE` before
starting each separately saved run; `all` enables every suite at once. Three seeds are
recommended for final comparisons. A smoke run verifies plumbing only.

This notebook refits all comparison heads fairly. Historical rank-96 weights and the
old colon-state cache are not required. Different models/checkpoints need their own
state collection and rank fitting; they cannot reuse this GPT-2 basis.